# MalIntent — DistilBERT Training Notebook
## Layer B: ML Classifier Fine-Tuning

**Project:** MalIntent — LLM Prompt Injection Firewall  
**Model:** distilbert-base-uncased  
**Dataset:** HackAPrompt (hackaprompt/hackaprompt-dataset)  
**Task:** Binary classification — prompt injection detection  
**Label mapping:** `0 = safe/benign (failed injection)` · `1 = confirmed attack (successful injection)`

## Objectives
- Load and inspect HackAPrompt dataset
- Clean, label, balance, and split into train / val / test sets
- Tokenize using DistilBERT tokenizer
- Fine-tune distilbert-base-uncased for binary sequence classification
- Evaluate on held-out test set and document metrics
- Save model weights for use in the MalIntent backend (Week 3+)

**Frameworks used:** HuggingFace Transformers · HuggingFace Datasets · Scikit-learn · Pandas · PyTorch

---
## Section 1: Environment Setup
Install dependencies and verify GPU availability. Run this cell at the start of every Colab session.

In [ ]:
# Cell 1 — Install dependencies
# Run this at the start of every Colab session — Colab resets between sessions

!pip install transformers datasets scikit-learn accelerate -q

import torch

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name:      {torch.cuda.get_device_name(0)}")
# Expected:
# GPU available: True
# GPU name:      Tesla T4
#
# If GPU available: False → Runtime → Change Runtime Type → T4 GPU → Save

---
## Section 2: Import Libraries

In [ ]:
# Cell 2 — All imports up front

import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)

print("All libraries imported successfully.")

---
## Section 3: Load HackAPrompt Dataset

HackAPrompt is a dataset of 600k+ real-world prompt injection attempts collected from a public competition where participants tried to jailbreak AI systems.  
Source: `hackaprompt/hackaprompt-dataset` on HuggingFace Hub.

In [ ]:
# Cell 3 — Load dataset
# No login required — this dataset is public

dataset = load_dataset("hackaprompt/hackaprompt-dataset")
df = pd.DataFrame(dataset['train'])

print("Dataset loaded successfully.")

---
## Section 4: Dataset Inspection
Always inspect before preprocessing — never assume column names or structure.

In [ ]:
# Cell 4 — Inspect dataset structure

print("Columns:", df.columns.tolist())
print("Shape:  ", df.shape)

print("\nSample row:")
print(df.iloc[0])

# The two columns that matter for labelling:
#   'user_input' — the text the attacker sent to the AI
#   'correct'    — True if the injection SUCCEEDED, False if it failed
print("\n--- Label source: 'correct' column ---")
print(df['correct'].value_counts())
print("\ncorrect=True  → confirmed injection attack  → label = 1")
print("correct=False → failed attempt / benign     → label = 0")

---
## Section 5: Preprocessing — Clean, Label, Balance, Split

**Labelling logic:**  
The `correct` column tells us whether an injection *worked*.  
- `correct=True`  → the attacker successfully manipulated the AI → **label = 1 (attack)**  
- `correct=False` → the attempt failed, the AI behaved correctly → **label = 0 (safe)**  

This is a real binary classification problem: can the model tell confirmed attacks from benign/failed inputs?

**Why balancing matters:**  
The raw dataset is heavily imbalanced (~524k safe, ~78k attacks). A model trained on this without balancing learns to always predict 'safe' and still gets ~87% accuracy — useless for a firewall. We cap the majority class to a 3:1 ratio to force genuine learning.

In [ ]:
# Cell 5 — Preprocess: clean, label, balance, split

# ── Step 1: Create binary labels ─────────────────────────────────────────────
# correct=True (injection succeeded)  → label 1 (attack)
# correct=False (injection failed)    → label 0 (safe/benign)
df['label'] = df['correct'].astype(int)

# ── Step 2: Keep only the two columns we need ─────────────────────────────────
# 'user_input' = the attacker's text (what we feed to the model)
# 'label'      = 0 or 1 (what the model learns to predict)
# NOTE: do NOT rename 'user_input' — all downstream tokenization code uses this name
df_clean = df[['user_input', 'label']].copy()

# ── Step 3: Remove nulls ──────────────────────────────────────────────────────
df_clean = df_clean.dropna(subset=['user_input'])

# ── Step 4: Remove empty / whitespace-only prompts ────────────────────────────
df_clean = df_clean[df_clean['user_input'].str.strip() != '']

# ── Step 5: Remove exact duplicates ──────────────────────────────────────────
df_clean = df_clean.drop_duplicates(subset=['user_input'])

# ── Step 6: Inspect class balance BEFORE balancing ───────────────────────────
print("=== Raw label distribution after cleaning ===")
print(df_clean['label'].value_counts())
print(f"Total examples: {len(df_clean)}")

attacks = df_clean[df_clean['label'] == 1]
safe    = df_clean[df_clean['label'] == 0]
print(f"\nAttacks (label=1): {len(attacks)}")
print(f"Safe    (label=0): {len(safe)}")

# ── Step 7: Balance — cap whichever class is larger to 3x the smaller ────────
# Case A: attacks >> safe (attacks outnumber safe by more than 3:1)
if len(attacks) > len(safe) * 3:
    attacks = attacks.sample(n=len(safe) * 3, random_state=42)
    print(f"\n[Balanced] Capped attacks to {len(attacks)}")

# Case B: safe >> attacks (safe examples outnumber attacks by more than 3:1)
elif len(safe) > len(attacks) * 3:
    safe = safe.sample(n=len(attacks) * 3, random_state=42)
    print(f"\n[Balanced] Capped safe examples to {len(safe)}")

# Recombine and shuffle
df_balanced = pd.concat([attacks, safe]).sample(frac=1, random_state=42)
print(f"\nBalanced dataset size: {len(df_balanced)}")
print("Balanced label distribution:")
print(df_balanced['label'].value_counts())

# ── Step 8: Three-way split — 80% train / 10% val / 10% test ─────────────────
# stratify= ensures the same attack/safe ratio in every split
# Without stratify, a random split could skew one set heavily toward one class
train_df, temp_df = train_test_split(
    df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label']
)

print(f"\nTrain : {len(train_df):>6} examples")
print(f"Val   : {len(val_df):>6} examples")
print(f"Test  : {len(test_df):>6} examples")

# Save test set — needed for Week 8 final evaluation
test_df.to_csv('test_set.csv', index=False)
print("\ntest_set.csv saved.")

---
## Section 6: Tokenization

DistilBERT cannot read raw text — it reads sequences of integer token IDs.  
The tokenizer converts each prompt into three tensors:

- **`input_ids`** — the integer IDs representing each token (word-piece)
- **`attention_mask`** — `1` for real tokens, `0` for padding (all inputs padded to the same length)
- **`label`** — `0` or `1`, passed through unchanged

**Why `max_length=128`?**  
DistilBERT's hard maximum is 512 tokens. But 512 uses 4× more GPU memory and trains 4× slower.  
Virtually all injection prompts in HackAPrompt are under 128 tokens, making 128 the right engineering choice.

In [ ]:
# Cell 6 — Tokenization

# DistilBertTokenizerFast = Rust-backed fast version
# Much faster than the standard tokenizer on large datasets
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_batch(batch):
    """Tokenize a batch of prompts into DistilBERT input format."""
    return tokenizer(
        batch['user_input'],    # the text column — must match DataFrame column name exactly
        truncation=True,        # cut off at max_length if longer
        padding='max_length',   # pad with zeros if shorter than max_length
        max_length=128,         # covers ~95% of prompts; 512 would be 4x slower
    )

# Convert pandas DataFrames → HuggingFace Dataset objects
# The Trainer API requires HuggingFace Datasets, not pandas DataFrames
train_dataset = Dataset.from_pandas(train_df[['user_input', 'label']])
val_dataset   = Dataset.from_pandas(val_df[['user_input', 'label']])
test_dataset  = Dataset.from_pandas(test_df[['user_input', 'label']])

# Apply tokenization in batches (batched=True processes many rows at once — much faster)
print("Tokenizing datasets... (may take 1-3 minutes)")
train_tokenized = train_dataset.map(tokenize_batch, batched=True)
val_tokenized   = val_dataset.map(tokenize_batch, batched=True)
test_tokenized  = test_dataset.map(tokenize_batch, batched=True)

# Convert to PyTorch tensor format
# Only keep the 3 columns the model actually uses — drop the raw 'user_input' text
cols = ['input_ids', 'attention_mask', 'label']
train_tokenized.set_format('torch', columns=cols)
val_tokenized.set_format('torch', columns=cols)
test_tokenized.set_format('torch', columns=cols)

# ── Verification ─────────────────────────────────────────────────────────────
print("\nFirst training example after tokenization:")
print(train_tokenized[0])
# Expected structure:
# {
#   'input_ids':      tensor([101, XXXX, XXXX, ..., 0, 0, 0])  ← 128 numbers
#   'attention_mask': tensor([1, 1, 1, ..., 0, 0, 0])           ← 1s then 0s
#   'label':          tensor(0) or tensor(1)
# }
# input_ids always starts with 101 = [CLS] token (mandatory for BERT models)
# Zeros at the end = padding; corresponding attention_mask values = 0

print(f"\nTrain tokenized : {len(train_tokenized):>6} examples")
print(f"Val   tokenized : {len(val_tokenized):>6} examples")
print(f"Test  tokenized : {len(test_tokenized):>6} examples")

---
## Section 7: Load Pre-trained DistilBERT Model

We load `distilbert-base-uncased` with a **classification head** added on top.  
The base model already understands English (trained by HuggingFace/Google on billions of words).  
Fine-tuning updates all ~66M parameters to specialise it for injection detection.

**Note on the expected warning:**  
You will see: *"Some weights were not initialized from the model checkpoint..."*  
This is **expected and correct** — the new classification head starts with random weights and will be trained from scratch.

In [ ]:
# Cell 7 — Load DistilBERT for sequence classification

# DistilBertForSequenceClassification = DistilBERT base + classification head
# The classification head is a small neural net that maps the model output → [safe_score, attack_score]
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2,   # 2 output classes: 0 = safe, 1 = attack
)

# Count total and trainable parameters (include this in your research paper)
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
# Expected:
# Total parameters     : 66,955,010
# Trainable parameters : 66,955,010
#
# All 66M parameters get updated — this is full fine-tuning.
# Research paper note: 'We fine-tuned distilbert-base-uncased (66M parameters)
# for binary sequence classification on the HackAPrompt benchmark.'

---
## Section 8: Define Evaluation Metrics

We track four metrics — not just accuracy:

| Metric | What it measures | Target |
|--------|-----------------|--------|
| **Accuracy** | % of all predictions correct | > 90% |
| **Precision** | Of flagged attacks, how many were real attacks? (low = too many false alarms) | > 80% |
| **Recall** | Of real attacks, how many did we catch? (low = attacks slipping through) | > 90% |
| **F1** | Harmonic mean of precision and recall | > 0.85 |

**For a security firewall, Recall matters more than Precision.**  
A missed attack (False Negative) is far worse than a false alarm (False Positive).

In [ ]:
# Cell 8 — Define metrics function
# Called automatically by the Trainer after each validation epoch

def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, and F1 from Trainer predictions."""
    logits, labels = eval_pred
    # logits = raw model output scores for each class
    # argmax picks the class with the higher score → predicted label
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions,
        average='binary',   # binary = treat as 2-class problem, report for positive class (attack=1)
        zero_division=0,    # avoids divide-by-zero if a class has no predictions
    )
    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy':  accuracy,
        'f1':        f1,
        'precision': precision,
        'recall':    recall,
    }

print("compute_metrics function defined.")

---
## Section 9: Configure Training Arguments

Every hyperparameter explained below. Start with these defaults.  
If F1 < 0.85 after training: try `num_train_epochs=5` or `learning_rate=3e-5`.

In [ ]:
# Cell 9 — Training configuration

training_args = TrainingArguments(
    output_dir='./malintent_model',         # where checkpoints are saved after each epoch

    # ── How long to train ────────────────────────────────────────────────────
    num_train_epochs=3,                     # start with 3; increase to 5 if F1 < 0.85

    # ── Batch sizes ──────────────────────────────────────────────────────────
    per_device_train_batch_size=32,         # 32 examples per GPU step during training
    per_device_eval_batch_size=64,          # 64 during evaluation (no gradients = more memory free)

    # ── Learning rate ────────────────────────────────────────────────────────
    learning_rate=2e-5,                     # 0.00002 — standard for BERT fine-tuning
    weight_decay=0.01,                      # L2 regularisation — prevents overfitting
    warmup_ratio=0.1,                       # LR slowly ramps up for the first 10% of steps
                                            # prevents large updates destroying pre-trained weights

    # ── Evaluation & checkpointing ───────────────────────────────────────────
    eval_strategy='epoch',                  # run validation after every epoch
    save_strategy='epoch',                  # save a checkpoint after every epoch
    load_best_model_at_end=True,            # after all epochs, restore the best checkpoint
    metric_for_best_model='f1',             # 'best' = highest F1 (NOT accuracy)

    # ── Logging ──────────────────────────────────────────────────────────────
    logging_steps=50,                       # print training loss every 50 steps
    report_to='none',                       # disable wandb/tensorboard

    # ── Reproducibility ──────────────────────────────────────────────────────
    seed=42,
)

print("TrainingArguments configured.")
print(f"Epochs          : {training_args.num_train_epochs}")
print(f"Train batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate   : {training_args.learning_rate}")

---
## Section 10: Train the Model

This cell starts the 2–3 hour training run.  
**Do not close your laptop or let the tab go idle — Colab will disconnect.**

**What to watch:**  
The `loss` value in the log output should decrease over time (from ~0.5–0.6 → ~0.1–0.2 by epoch 3).  
If loss stays flat or increases, the learning rate may be too high.

In [ ]:
# Cell 10 — Build Trainer and START TRAINING
# Expected time: 2–3 hours on T4 GPU

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,       # validated after each epoch — needs val split
    compute_metrics=compute_metrics,
)

print("Starting training... (2–3 hours on T4 GPU)")
print("Expected log output: {'loss': 0.45, 'learning_rate': 1.8e-05, 'epoch': 0.12}")
print("Loss should decrease each epoch. If it doesn't, see troubleshooting below.")
print("-" * 60)

train_result = trainer.train()

print("\n=" * 30)
print("TRAINING COMPLETE")
print("=" * 30)
print(f"Final training loss : {train_result.training_loss:.4f}")
print(train_result.metrics)

---
## Section 11: Save Model Weights

**Run this cell IMMEDIATELY after training completes.**  
If Colab disconnects before saving, you lose everything and must retrain from scratch.  
Save to Google Drive first (persistent), then download to your laptop.

In [ ]:
# Cell 11 — Save model weights IMMEDIATELY after training
from google.colab import drive

# Mount Google Drive (you'll be prompted to authenticate)
drive.mount('/content/drive')

# Save to Google Drive — persistent across sessions
save_path = '/content/drive/MyDrive/malintent_model'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to Google Drive: {save_path}")

# Also save locally (for downloading to your laptop)
trainer.save_model('./malintent_model_local')
tokenizer.save_pretrained('./malintent_model_local')
print("Model saved locally: ./malintent_model_local")

# List saved files
print("\nFiles in Drive save:")
!ls {save_path}
# Expected: config.json  model.safetensors  tokenizer files
# model.safetensors is ~250MB — this is the main weights file

print("\n--- NEXT STEP ---")
print("Download malintent_model_local/ from the Files panel (left sidebar → right-click → Download)")
print("Place it in your repo at: backend/malintent_model_local/")

---
## Section 12: Final Evaluation on Test Set

Validation during training used the val set.  
This is the **final evaluation on the test set** — data the model has never seen at all.  
These are the numbers you report in your research paper.

**Targets:**

| Metric | Target | If Below — Try This |
|--------|--------|---------------------|
| Accuracy | > 90% | Check class balance |
| Precision | > 80% | Adjust decision threshold |
| Recall | > 90% | Train more epochs or check labels |
| F1 Score | > 0.85 | Try 5 epochs or `learning_rate=3e-5` |

In [ ]:
# Cell 12 — Final test set evaluation

# Run predictions on the held-out test set
predictions_output = trainer.predict(test_tokenized)
logits = predictions_output.predictions
labels = predictions_output.label_ids
preds  = np.argmax(logits, axis=-1)

# Full classification report
print("=== TEST SET EVALUATION ===")
print(classification_report(labels, preds, target_names=['safe', 'attack']))

# Confusion matrix — formatted for readability
cm = confusion_matrix(labels, preds)
print("Confusion Matrix:")
print("                   Predicted Safe   Predicted Attack")
print(f"Actual Safe   :        {cm[0][0]:>6}             {cm[0][1]:>6}")
print(f"Actual Attack :        {cm[1][0]:>6}             {cm[1][1]:>6}")
print()
print(f"True Negatives  (correctly safe)  : {cm[0][0]}")
print(f"False Positives (false alarms)    : {cm[0][1]}")
print(f"False Negatives (MISSED ATTACKS!) : {cm[1][0]}  ← minimise this")
print(f"True Positives  (caught attacks)  : {cm[1][1]}")

---
## Section 13: Troubleshooting — If F1 < 0.85

Don't retrain blindly. Work through this checklist in order:

1. **Try 5 epochs** — change `num_train_epochs=5`. Most common fix.
2. **Try a higher learning rate** — change `learning_rate=3e-5`.
3. **Verify labels** — run `df_balanced['label'].value_counts()`. If 95%+ is one class, the balancing logic has a bug.
4. **Check column name** — make sure `batch['user_input']` in `tokenize_batch` matches your actual DataFrame column name.
5. **Try max_length=256** — some longer injection prompts get truncated at 128 and lose important context.

F1 of 0.82–0.84 is still excellent and publishable — document honestly in your paper.

---
## Section 14: Final Commit Instructions

Run the following in your terminal (not in Colab) after downloading the model:

```bash
# From your backend/ folder
git add .
git commit -m "feat: Layer B DistilBERT classifier, F1=[YOUR SCORE], training notebook"
git push origin main
```

**Files to commit:**
- `backend/notebooks/malintent_distilbert_training.ipynb` — this notebook
- `backend/malintent_model_local/` — the downloaded model weights folder
- `docs/model_evaluation.md` — your metrics table (fill in after evaluation)

**Do NOT commit `model.safetensors` directly to GitHub** — it's ~250MB and will be rejected.  
Add it to `.gitignore` and keep it in Google Drive + your local machine only.